### Imports

In [ ]:
import sys
import numpy as np
import pytest
import matplotlib.pyplot as plt
sys.path.append("../src")

from system import System
from polymer import Polymer

from mpcd import *
from md import verlet_step
from forces import forcesOnPolymer
from observables import * 

### Parameters of Simulation

In [2]:
params = {
    "box": np.array([10.0, 10.0, 10.0]),

    # MPCD solvent
    "n_solvent": 10000,
    "solvent_mass": 1.0,
    "a": 1.0,
    "h": 0.1,
    "alpha": np.pi / 2,
    "kBT": 1.0,

    # Polymer
    "n_monomers": 100,
    "polymer_mass": 3.0,
    "bond_length": 0.25,
    "k_bond": 100.0,

    # MD
    "n_md_substeps": 1000,
    "n_mpcd_steps": 500,

    # Random seed
    "seed": 91109,
}
dt_md = params["h"] / params["n_md_substeps"]

### Creation of a Polymer and a System

In [3]:
system = System(
    N=params["n_solvent"],
    box=params["box"],
    m=params["solvent_mass"],
    a=params["a"],
    h=params["h"],
    alpha=params["alpha"],
    kBT=params["kBT"],
    seed=params["seed"],
)

polymer = Polymer(
    nMonomers=params["n_monomers"],
    bondLength=params["bond_length"],
    dt=dt_md,
    m=params["polymer_mass"],
    k=params["k_bond"],
    box=params["box"],
    kBT=params["kBT"],
    seed=params["seed"] + 1,
)

forcesOnPolymer(polymer)

### Simulation

In [5]:
history = {
    "step": [],
    "polymer_E": [],
    "polymer_K": [],
    "polymer_U": [],
    "mean_bond_length": [],
    "max_bond_length": [],
    "polymer_momentum_norm": [],
    "solvent_momentum_norm": [],
}

P_polymer_0 = total_momentum(polymer)
P_solvent_0 = total_momentum(system)

for step in range(params["n_mpcd_steps"]):
    # 1. MPCD streaming
    stream(system)

    # 2. Polymer MD substeps
    for _ in range(params["n_md_substeps"]):
        verlet_step(polymer, dt_md, forcesOnPolymer)

    # 3. MPCD collision
    # for now:
    collide(system)

    # collide_with_polymer(system, polymer)

    # 4. Record observables
    lengths = bond_lengths(polymer)

    history["step"].append(step)
    history["polymer_K"].append(polymer_kinetic_energy(polymer))
    history["polymer_U"].append(polymer_bond_energy(polymer))
    history["polymer_E"].append(polymer_total_energy(polymer))
    history["mean_bond_length"].append(lengths.mean())
    history["max_bond_length"].append(lengths.max())
    history["polymer_momentum_norm"].append(
        np.linalg.norm(total_momentum(polymer) - P_polymer_0)
    )
    history["solvent_momentum_norm"].append(
        np.linalg.norm(total_momentum(system) - P_solvent_0)
    )
    
for key in history:
    history[key] = np.array(history[key])

### Results
#### Polymer

In [6]:
for key in history:
    history[key] = np.array(history[key])

In [8]:
plt.figure()
plt.plot(history["step"], history["polymer_E"])
plt.xlabel("MPCD step")
plt.ylabel("Polymer total energy")
plt.title("Polymer total energy")
plt.show()

AttributeError: module 'matplotlib' has no attribute 'figure'